# 21 · Real batting features, round 2 (feature groups from published work)

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

Round 1 plus the groups found in cricket analytics writing: win chance added and wicket-cost runs per situation, back-to-back dots, sixes, boundaries per dismissal, dismissal type, spin turning away or in by batting hand, collapse entry, chases above 9 an over, outcomes on attacking and defensive strokes, shot variety and on-side scoring (about 250 features). Same splits, models and rule.

Result: the current rating still scores best (0.143 against 0.134 for the best feature model), so it stays.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P24b - Round 2: the P24 test with the extra feature groups from published work (research agent report, 2026-11-14):  
win chance added and wicket-cost-inclusive runs per situation, back-to-back dots, sixes, boundaries per dismissal, dismissal  
mix, spin turning away / in by batting hand, collapse entry, chases above 9 an over, outcomes on attacking / defensive  
strokes, shot variety, on-side share, positive-WPA innings share. Same splits, models and pre-registered rule as P24.  
P24 - Do real cricket batting features predict pressure performance better than the current batting rating? (research)  
  
Rule fixed BEFORE running (2026-11-14):  
  Features: ~150 per batter, computed ONLY from the training half of each split. For 24 situations (pressure, phases,  
  chasing, wickets down, new or set batter, pace / spin, yorkers, short balls, full tosses, slower balls, wrist / finger  
  spin, and pressure combinations) his situation-adjusted performance: runs, boundaries, dots, dismissals and running  
  (1s/2s/3s), each as 'actual minus an average batter in the same spot', shrunk towards 0 for small samples  
  (n / (n + 30)), plus how many balls he faced there. Also shot style (aggressive / defensive share, against each year's  
  average), match-to-match consistency and experience. Swing / seam labels are not used (inconsistent across years).  
  Target: the P9 yardstick, his win chance added per pressure ball in the hidden half (>= 20 pressure balls).  
  Models, cross-fitted over players within each split (5 folds, so no player predicts himself):  
    ridge, ridge + current rating, elastic net (feature selection), small LightGBM.  
  Compared with the current rating on the same pool, same 60-pressure-ball floor, same 52 splits as P9.  
  A model replaces the current rating only if its objective (0.5 random + 0.5 rolling rank correlation) beats it by  
  more than one paired standard error. Only then is it opened on the 3,993 outside matches.  
Nothing existing is overwritten: results go to research_batfeat_* tables.
</details>

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os; os.environ["PYTHONWARNINGS"] = "ignore"
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import lightgbm as lgb
from joblib import Parallel, delayed
from rrlib.common import CFG, SEED, load, load_json, save_table, save_json, get_logger, Timer, weighted_spearman
from rrlib import ratings as RL

log = get_logger("p24b_batting_features")
v = load("values")
f = load("features", columns=["ball_uid", "over_no", "wkts_before", "str_balls", "pace_or_spin", "delivery_style", "stroke", "y_class",
                              "batting_type", "bowling_type", "on_or_off", "rrr", "dismissal_type"])
c = load("ctx_preds", columns=["ball_uid", "p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt"])
assert (v.ball_uid.values == f.ball_uid.values).all() and (v.ball_uid.values == c.ball_uid.values).all()
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
FLOOR = chosen.get("min_n", 60)

# ---------------------------------------------------------------- ball table for batters
pb = c[["p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt"]].values
pb = pb / pb.sum(1, keepdims=True)
B = v[["match_id", "year", "level", "striker_id", "is_pressure", "innings", "phase", "runs", "boundary", "striker_out",
       "v1_bat", "v2_bat", "v3_bat", "r_bdry_bat", "r_dot_bat", "r_wkt_bat", "r_runs_bat", "faced"]].copy()
B["r_run_bat"] = np.where(np.isin(f.y_class.values, [1, 2, 3]), v.runs.values, 0.0) - (pb[:, 1] + 2 * pb[:, 2] + 3 * pb[:, 3])
B["r_six_bat"] = (v.runs.values == 6).astype(float) - pb[:, 5]
B["p_wkt"] = pb[:, 6]
for col in ("over_no", "wkts_before", "str_balls", "pace_or_spin", "delivery_style", "stroke", "batting_type", "bowling_type", "on_or_off", "rrr", "dismissal_type"):
    B[col] = f[col].values
B = B[B.faced.astype(bool)].reset_index(drop=True)
B["prev_dot"] = (B.runs.eq(0) & B.striker_out.eq(0)).astype(int).groupby([B.match_id, B.innings, B.striker_id]).shift(1).fillna(0).astype(bool)
bt, bw = B.batting_type.fillna(""), B.bowling_type.fillna("")
away_r = bw.isin(["Right-arm Leg-Break", "Slow left-arm orthodox"]); in_r = bw.isin(["Right-arm offbreak", "Left-arm Chinaman"])
B["is_pressure"] = B.is_pressure.astype(bool)
ds, st = B.delivery_style.fillna(""), B.stroke.fillna("")
AGG = {"Slog", "Slog Sweep", "Pull shot", "Hook Shot", "UpperCut", "Reverse Sweep", "Paddle Scoop", "Reverse Paddle Scoop", "Switch",
       "PickUp", "Inside out"}
DEF = {"Frontfoot defensive", "Backfoot defensive", "Defense", "Push"}
B["sh_agg"] = st.isin(AGG).astype(float)
B["sh_def"] = st.isin(DEF).astype(float)
for sc_ in ("sh_agg", "sh_def"):                      # against each year's own average share (labels drift over time)
    B[sc_] = B[sc_] - B.groupby("year")[sc_].transform("mean")
CTX = {
    "all": np.ones(len(B), bool), "pressure": B.is_pressure.values, "nonpressure": ~B.is_pressure.values,
    "powerplay": B.over_no.between(1, 6).values, "middle": B.over_no.between(7, 15).values, "death": B.over_no.between(16, 20).values,
    "chase": (B.innings == 2).values, "setting": (B.innings == 1).values, "wk4plus": (B.wkts_before >= 4).values,
    "new_batter": (B.str_balls < 10).values, "set_batter": (B.str_balls >= 20).values,
    "pace": (B.pace_or_spin == "Pace").values, "spin": (B.pace_or_spin == "Spin").values,
    "yorker": ds.isin(["Yorker", "Wide Yorker"]).values, "short": ds.isin(["Bouncer", "Slow Bouncer"]).values,
    "fulltoss": ds.eq("Full Toss").values,
    "slower": ds.isin(["Off Cutter", "Leg Cut", "Knuckle", "Slow Ball", "Slow Bouncer"]).values,
    "wrist_spin": ds.isin(["Leg Break", "Googly", "Chinaman", "Back of Hand", "Flipper"]).values,
    "finger_spin": ds.isin(["Off Break", "Orthodox", "Arm ball", "Carrom"]).values,
    "mid_chase_wk3": (B.over_no.between(7, 15) & (B.innings == 2) & (B.wkts_before >= 3)).values,
    "death_chase": (B.over_no.between(16, 20) & (B.innings == 2)).values,
    "pressure_spin": (B.is_pressure & (B.pace_or_spin == "Spin")).values,
    "pressure_pace": (B.is_pressure & (B.pace_or_spin == "Pace")).values,
    "pressure_death": (B.is_pressure & B.over_no.between(16, 20)).values,
    "after_dot": B.prev_dot.values,
    "spin_away": ((bt.eq("RHB") & away_r) | (bt.eq("LHB") & in_r)).values,
    "spin_in": ((bt.eq("RHB") & in_r) | (bt.eq("LHB") & away_r)).values,
    "collapse_entry": ((B.str_balls < 20) & (B.wkts_before >= 3) & (B.over_no <= 10)).values,
    "chase_rrr9": ((B.innings == 2) & (B.rrr > 9)).values,
    "attacking_shot": B.sh_agg.gt(0).values, "defensive_shot": B.sh_def.gt(0).values}
OUTS = {"runs": "r_runs_bat", "bdry": "r_bdry_bat", "dot": "r_dot_bat", "out": "r_wkt_bat", "run": "r_run_bat",
        "runs_incl_wicket_cost": "v1_bat", "win_chance": "v2_bat", "six": "r_six_bat"}
K_PRIOR = 30.0
for k_, m_ in CTX.items():
    B[f"m_{k_}"] = m_


def features(T):
    """per-batter features from a ball table T (training matches only)"""
    g = T.groupby("striker_id")
    F = pd.DataFrame(index=g.size().index)
    for k_ in CTX:
        sub = T[T[f"m_{k_}"]]
        gs = sub.groupby("striker_id")
        n = gs.size().reindex(F.index).fillna(0)
        F[f"n_{k_}"] = np.log1p(n)
        for o, col in OUTS.items():
            s = gs[col].sum().reindex(F.index).fillna(0)
            F[f"{k_}_{o}"] = s / (n + K_PRIOR)           # shrunk mean of 'actual - expected'
    for sc_ in ("sh_agg", "sh_def"):
        F[sc_] = g[sc_].sum() / (g.size() + K_PRIOR)
    inn = T.groupby(["striker_id", "match_id"]).r_runs_bat.agg(["sum", "size"])
    inn = inn[inn["size"] >= 5]
    ig = (inn["sum"] / inn["size"]).groupby(level=0)
    F["cons_pos_share"] = ig.apply(lambda x: (x > 0).mean()).reindex(F.index).fillna(0.5)
    F["cons_sd"] = ig.std().reindex(F.index).fillna(ig.std().median())
    F["n_matches"] = np.log1p(T.groupby("striker_id").match_id.nunique())
    F["ipl_share"] = g.apply(lambda d: (d.level == "IPL").mean())
    bd = g.boundary.sum(); outs = g.striker_out.sum()
    F["bdry_per_dismissal"] = np.log((bd + 2) / (outs + 1))
    P = T[T.is_pressure].groupby("striker_id")
    F["bdry_per_dismissal_pressure"] = np.log((P.boundary.sum() + 2) / (P.striker_out.sum() + 1)).reindex(F.index)
    dt = T[T.striker_out.eq(1)].groupby("striker_id").dismissal_type
    beaten = dt.apply(lambda x: x.isin(["Bowled", "LBW"]).sum()).reindex(F.index).fillna(0)
    F["beaten_share"] = (beaten + 0.3) / (outs + 1)
    F["risk_faced"] = g.p_wkt.mean()
    F["risk_faced_pressure"] = P.p_wkt.mean().reindex(F.index)
    on = T[T.on_or_off.eq("On-side")].groupby("striker_id").runs.sum().reindex(F.index).fillna(0)
    F["onside_run_share"] = (on + 10) / (g.runs.sum() + 20)
    def ent(x):
        p_ = x.value_counts(normalize=True).values
        return float(-(p_ * np.log(p_)).sum())
    F["shot_variety"] = g.stroke.apply(ent)
    innw = T.groupby(["striker_id", "match_id"]).v2_bat.sum()
    F["pos_winchance_innings"] = innw.gt(0).groupby(level=0).mean().reindex(F.index)
    return F.fillna(F.median())


# ---------------------------------------------------------------- splits (as P9)
HOLD = CFG["cv"]["holdout_season"]
V = v[v.year < HOLD]
splits = []
for tr_years, te_year in CFG["cv"]["rolling"]:
    splits.append(("rolling", f"{tr_years[0]}-{tr_years[-1]}->{te_year}", V.year.isin(tr_years).values, (V.year == te_year).values))
mids = np.sort(V.match_id.unique())
rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False))
    m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))


def one(kind, name, tr_ids, te_ids, vtrain):
    Btr, Bte = B[B.match_id.isin(tr_ids)], B[B.match_id.isin(te_ids)]
    t = Bte[Bte.is_pressure & Bte.v2_bat.notna()]
    y = t.groupby("striker_id").agg(y2=("v2_bat", "mean"), y1=("v1_bat", "mean"), n=("v2_bat", "size"))
    y = y[y.n >= 20]
    np_tr = Btr[Btr.is_pressure].groupby("striker_id").size()
    pool = y.index[y.index.isin(np_tr.index)]
    if len(pool) < 15:
        return None
    y = y.loc[pool]
    eligible = (np_tr.reindex(pool).fillna(0) >= FLOOR).values
    F = features(Btr).reindex(pool).fillna(0.0)
    cur = RL.build_scores(vtrain, "bat", chosen, comp).score.reindex(pool)
    F_plus = F.assign(current=cur.fillna(cur.min() if cur.notna().any() else 0).values)
    preds = {"current": cur.fillna(-1e9).values}
    out = {k: np.zeros(len(pool)) for k in ("ridge", "ridge_plus_current", "elastic_net", "lightgbm")}
    coefs = []
    kf = KFold(5, shuffle=True, random_state=SEED)
    w = y.n.values.astype(float)
    for tr_i, te_i in kf.split(pool):
        for key, X in (("ridge", F), ("ridge_plus_current", F_plus)):
            sc = StandardScaler().fit(X.values[tr_i])
            mdl = RidgeCV(alphas=np.logspace(-1, 4, 20)).fit(sc.transform(X.values[tr_i]), y.y2.values[tr_i], sample_weight=w[tr_i])
            out[key][te_i] = mdl.predict(sc.transform(X.values[te_i]))
        sc = StandardScaler().fit(F.values[tr_i])
        en = ElasticNetCV(l1_ratio=0.5, alphas=np.logspace(-4, -0.5, 30), cv=4, max_iter=5000, random_state=SEED).fit(sc.transform(F.values[tr_i]), y.y2.values[tr_i])
        out["elastic_net"][te_i] = en.predict(sc.transform(F.values[te_i]))
        coefs.append(pd.Series(en.coef_, index=F.columns))
        gbm = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=20, subsample=0.8,
                                subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, random_state=SEED, verbose=-1)
        gbm.fit(F.values[tr_i], y.y2.values[tr_i], sample_weight=w[tr_i])
        out["lightgbm"][te_i] = gbm.predict(F.values[te_i])
    for k_, p_ in out.items():
        preds[k_] = np.where(eligible, p_, -1e9)       # same 60-pressure-ball floor as the current rating
    rows = []
    for k_, p_ in preds.items():
        top = np.argsort(-p_)[:10]
        wm = lambda col, idx: np.average(y[col].values[idx], weights=w[idx])
        rows.append({"kind": kind, "split": name, "model": k_, "n_pool": len(pool),
                     "rho_v2": weighted_spearman(p_, y.y2.values, w), "rho_v1": weighted_spearman(p_, y.y1.values, w),
                     "top10_lift_v1": 100 * (wm("y1", top) - np.average(y.y1.values, weights=w)),
                     "top10_lift_v2": 100 * (wm("y2", top) - np.average(y.y2.values, weights=w))})
    sel = (pd.concat(coefs, axis=1).abs() > 1e-9).mean(1)
    return rows, sel.rename(name)


with Timer(log, "52 splits: features + 4 models"):
    jobs = [(k, n, set(V.match_id[tr]), set(V.match_id[te]), V[tr]) for k, n, tr, te in splits]
    res = Parallel(n_jobs=min(8, CFG["n_jobs"]))(delayed(one)(*j) for j in jobs)
res = [r for r in res if r is not None]
R = pd.DataFrame([row for rows, _ in res for row in rows])
SEL = pd.concat([s for _, s in res], axis=1)
save_table(R, "research_batfeat2_cv")
selection = SEL.mean(1).sort_values(ascending=False)
save_table(selection.rename("share_of_fits_selected").reset_index().rename(columns={"index": "feature"}), "research_batfeat2_selection")
summ = {}
for m_ in R.model.unique():
    r = R[R.model == m_]
    summ[m_] = {"objective": float(0.5 * r[r.kind == "random"].rho_v2.mean() + 0.5 * r[r.kind == "rolling"].rho_v2.mean()),
                "rolling_rho_v2": float(r[r.kind == "rolling"].rho_v2.mean()), "rho_v1": float(r.rho_v1.mean()),
                "top10_lift_v1": float(r.top10_lift_v1.mean()), "top10_lift_v2": float(r.top10_lift_v2.mean())}
piv = R[R.kind == "random"].pivot(index="split", columns="model", values="rho_v2")
for m_ in piv.columns:
    if m_ == "current":
        continue
    dd = piv[m_] - piv["current"]
    summ[m_].update({"vs_current_diff": float(dd.mean()), "vs_current_se": float(dd.std(ddof=1) / np.sqrt(len(dd))),
                     "better_splits": int((dd > 0).sum()), "objective_gain": summ[m_]["objective"] - summ["current"]["objective"]})
best = max((m_ for m_ in summ if m_ != "current"), key=lambda m_: summ[m_]["objective"])
summ["best_model"] = best
summ["decision"] = ("adopt " + best) if (summ[best]["objective_gain"] > summ[best]["vs_current_se"] and
                                        summ[best]["rolling_rho_v2"] >= summ["current"]["rolling_rho_v2"]) else "keep current rating"
save_json(summ, "research_batfeat2_summary")
log.info("\n" + pd.DataFrame({k: v_ for k, v_ in summ.items() if isinstance(v_, dict)}).round(4).to_string())
log.info(f"decision by the pre-registered rule: {summ['decision']}")
log.info("features selected most often by the elastic net (share of fits):\n" + selection.head(25).round(2).to_string())


22:28:46 [52 splits: features + 4 models] 103.1s


22:28:46 
                 current    ridge  ridge_plus_current  elastic_net  lightgbm
objective         0.1427   0.1335              0.1332       0.0848    0.1248
rolling_rho_v2    0.1500   0.1294              0.1287       0.0747    0.1128
rho_v1            0.1510   0.1300              0.1304       0.1045    0.1320
top10_lift_v1    11.6837   6.5957              6.3897       2.5757    5.6501
top10_lift_v2     0.2672   0.2497              0.2441       0.0540    0.2057
vs_current_diff      NaN   0.0023              0.0022      -0.0405    0.0014
vs_current_se        NaN   0.0051              0.0051       0.0055    0.0053
better_splits        NaN  29.0000             29.0000       6.0000   21.0000
objective_gain       NaN  -0.0092             -0.0096      -0.0579   -0.0179


22:28:46 decision by the pre-registered rule: keep current rating


22:28:46 features selected most often by the elastic net (share of fits):
wrist_spin_win_chance             0.13
yorker_win_chance                 0.11
fulltoss_runs                     0.11
n_yorker                          0.08
fulltoss_win_chance               0.07
yorker_dot                        0.07
chase_rrr9_win_chance             0.07
short_run                         0.07
fulltoss_six                      0.06
n_pressure_death                  0.06
n_mid_chase_wk3                   0.05
yorker_out                        0.05
pressure_death_dot                0.05
risk_faced                        0.05
slower_win_chance                 0.05
sh_agg                            0.04
fulltoss_runs_incl_wicket_cost    0.04
chase_rrr9_out                    0.03
n_pressure_spin                   0.03
collapse_entry_dot                0.03
wk4plus_dot                       0.03
sh_def                            0.03
short_win_chance                  0.03
yorker_six                   